# Phase 2 — Deep Learning Model Training
**XAI Healthcare Diagnostics Project**

This notebook trains three models:
1. **ResNet-50** — CNN baseline
2. **EfficientNet-B3** — Lightweight CNN
3. **ViT-B/16** — Vision Transformer

On two datasets:
- **ISIC 2019** — Skin lesion classification (8 classes)
- **ChestX-ray14** — Chest disease classification (14 labels)

---
> **Run on Google Colab with T4 GPU for best results.**

## 0. Colab Setup

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Update this to where you uploaded xai_project on your Drive
PROJECT_ROOT = '/content/drive/MyDrive/xai_project'
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install timm albumentations torchmetrics -q

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Imports & Config

In [ ]:
import sys
sys.path.append('.')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
import torchvision.models as models
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, classification_report

from src.preprocessing.preprocess import (
    ISICDataset, ChestXrayDataset,
    get_train_transforms, get_val_transforms,
    ISIC_LABELS, CHESTXRAY_LABELS,
    split_isic, split_chestxray,
)

# ── Paths ─────────────────────────────────────────────────────
CHESTXRAY_ROOT = 'data/chestxray'
ISIC_ROOT      = 'data/isic'
MODELS_DIR     = Path('outputs/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Training Config ───────────────────────────────────────────
BATCH_SIZE  = 32
NUM_EPOCHS  = 15
LR          = 1e-4
NUM_WORKERS = 2
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config ready.')

## 2. Load ISIC Dataset

In [ ]:
# ── Build ISIC DataFrame (fixed version) ─────────────────────
def build_isic_df_fixed(data_root):
    from pathlib import Path
    import numpy as np
    import pandas as pd
    ISIC_LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
    data_root = Path(data_root)
    gt = pd.read_csv(data_root / 'ISIC_2019_Training_GroundTruth.csv')
    img_dir = data_root / 'ISIC_2019_Training_Input' / 'ISIC_2019_Training_Input'
    records = []
    for _, row in gt.iterrows():
        img_path = img_dir / f"{row['image']}.jpg"
        if not img_path.exists():
            continue
        label_idx = int(np.argmax(row[ISIC_LABELS].values))
        records.append({
            'image_path': str(img_path),
            'label': label_idx,
            'class_name': ISIC_LABELS[label_idx],
        })
    return pd.DataFrame(records)

df_is = build_isic_df_fixed(ISIC_ROOT)
train_df, val_df, test_df = split_isic(df_is)

# ── Compute class weights for imbalanced data ─────────────────
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = torch.tensor(
    1.0 / class_counts * class_counts.sum() / len(class_counts),
    dtype=torch.float32
).to(DEVICE)

# ── DataLoaders ───────────────────────────────────────────────
train_ds = ISICDataset(train_df, transform=get_train_transforms())
val_ds   = ISICDataset(val_df,   transform=get_val_transforms())
test_ds  = ISICDataset(test_df,  transform=get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
print(f'Class weights: {class_weights.cpu().numpy().round(2)}')

## 3. Model Definitions

In [ ]:
def get_model(model_name: str, num_classes: int, pretrained: bool = True):
    """
    Returns a pretrained model with the final layer replaced.
    Supports: resnet50, efficientnet_b3, vit_b16
    """
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == 'efficientnet_b3':
        model = timm.create_model('efficientnet_b3', pretrained=pretrained, num_classes=num_classes)

    elif model_name == 'vit_b16':
        model = timm.create_model('vit_base_patch16_224', pretrained=pretrained, num_classes=num_classes)

    else:
        raise ValueError(f'Unknown model: {model_name}')

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'{model_name}: {total_params:.1f}M parameters')
    return model.to(DEVICE)


# Preview all three
for name in ['resnet50', 'efficientnet_b3', 'vit_b16']:
    m = get_model(name, num_classes=8)
    del m
    torch.cuda.empty_cache()

## 4. Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    criterion = nn.CrossEntropyLoss()
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    try:
        auc = roc_auc_score(
            np.eye(8)[all_labels], all_probs, multi_class='ovr', average='macro'
        )
    except:
        auc = 0.0
    return total_loss / total, correct / total, auc


def train_model(model_name, num_classes, train_loader, val_loader,
                num_epochs=NUM_EPOCHS, class_weights=None):
    print(f'\n{"="*55}')
    print(f'  Training: {model_name}')
    print(f'{"="*55}')

    model = get_model(model_name, num_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_auc':[]}
    best_auc = 0.0
    best_path = MODELS_DIR / f'{model_name}_best.pth'

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, vl_auc = evaluate(model, val_loader)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['val_auc'].append(vl_auc)

        print(f'Epoch [{epoch:02d}/{num_epochs}]  '
              f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  '
              f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  AUC: {vl_auc:.4f}')

        # Save best model
        if vl_auc > best_auc:
            best_auc = vl_auc
            torch.save({
                'epoch': epoch,
                'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_auc,
                'num_classes': num_classes,
            }, best_path)
            print(f'  ✓ Best model saved (AUC: {best_auc:.4f})')

    print(f'\n✓ Training complete. Best Val AUC: {best_auc:.4f}')
    print(f'  Model saved to: {best_path}')
    return model, history


def plot_history(history, model_name, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0].plot(epochs, history['val_loss'],   'r-', label='Val')
    axes[0].set_title(f'{model_name} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], 'b-', label='Train')
    axes[1].plot(epochs, history['val_acc'],   'r-', label='Val')
    axes[1].set_title(f'{model_name} — Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].legend()

    axes[2].plot(epochs, history['val_auc'], 'g-', label='Val AUC')
    axes[2].set_title(f'{model_name} — AUC-ROC')
    axes[2].set_xlabel('Epoch'); axes[2].legend()

    plt.suptitle(f'{model_name} Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

print('Functions ready.')

## 5. Train ResNet-50 on ISIC

In [ ]:
resnet_model, resnet_history = train_model(
    model_name    = 'resnet50',
    num_classes   = 8,
    train_loader  = train_loader,
    val_loader    = val_loader,
    num_epochs    = NUM_EPOCHS,
    class_weights = class_weights,
)
plot_history(resnet_history, 'ResNet-50',
             save_path='outputs/reports/resnet50_isic_history.png')

## 6. Train EfficientNet-B3 on ISIC

In [ ]:
effnet_model, effnet_history = train_model(
    model_name    = 'efficientnet_b3',
    num_classes   = 8,
    train_loader  = train_loader,
    val_loader    = val_loader,
    num_epochs    = NUM_EPOCHS,
    class_weights = class_weights,
)
plot_history(effnet_history, 'EfficientNet-B3',
             save_path='outputs/reports/efficientnet_isic_history.png')

## 7. Train ViT-B/16 on ISIC

In [ ]:
vit_model, vit_history = train_model(
    model_name    = 'vit_b16',
    num_classes   = 8,
    train_loader  = train_loader,
    val_loader    = val_loader,
    num_epochs    = NUM_EPOCHS,
    class_weights = class_weights,
)
plot_history(vit_history, 'ViT-B/16',
             save_path='outputs/reports/vit_isic_history.png')

## 8. Compare All Three Models

In [ ]:
# Load best checkpoints and evaluate on test set
results = {}

for model_name in ['resnet50', 'efficientnet_b3', 'vit_b16']:
    ckpt = torch.load(MODELS_DIR / f'{model_name}_best.pth', map_location=DEVICE)
    model = get_model(model_name, num_classes=8, pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_auc = evaluate(model, test_loader)
    results[model_name] = {
        'Test Accuracy': round(test_acc * 100, 2),
        'Test AUC-ROC':  round(test_auc, 4),
        'Best Val AUC':  round(ckpt['best_auc'], 4),
        'Best Epoch':    ckpt['epoch'],
    }
    print(f'{model_name:<20} Acc: {test_acc:.4f}  AUC: {test_auc:.4f}')

results_df = pd.DataFrame(results).T
print('\n── Final Model Comparison ──────────────────────')
print(results_df.to_string())
results_df.to_csv('outputs/reports/model_comparison_isic.csv')

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
models_list = list(results.keys())
accs = [results[m]['Test Accuracy'] for m in models_list]
aucs = [results[m]['Test AUC-ROC']  for m in models_list]
colors = ['#4C72B0', '#DD8452', '#55A868']

axes[0].bar(models_list, accs, color=colors)
axes[0].set_title('Test Accuracy (%)', fontweight='bold')
axes[0].set_ylim(0, 100)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

axes[1].bar(models_list, aucs, color=colors)
axes[1].set_title('Test AUC-ROC', fontweight='bold')
axes[1].set_ylim(0, 1)
for i, v in enumerate(aucs):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.suptitle('ISIC 2019 — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/reports/model_comparison_isic.png', dpi=150)
plt.show()
print('✓ Comparison saved')

## 9. Select Best Model & Save for Phase 3
The best performing model will be used for XAI in Phase 3.

In [ ]:
# Automatically select best model by AUC
best_model_name = max(results, key=lambda x: results[x]['Test AUC-ROC'])
print(f'\n✓ Best model: {best_model_name}')
print(f'  Test AUC-ROC : {results[best_model_name]["Test AUC-ROC"]}')
print(f'  Test Accuracy: {results[best_model_name]["Test Accuracy"]}%')
print(f'\nCheckpoint: outputs/models/{best_model_name}_best.pth')
print('\n✓ Phase 2 complete! Download outputs/models/ and use in Phase 3.')